<a href="https://colab.research.google.com/github/ldongheedev/-BDA-LLM-RAG-Program/blob/main/3%EC%A3%BC%EC%B0%A8_TF_IDF_9%EB%8B%A8%EA%B3%84_%EB%8F%84%EC%A0%84%EA%B3%BC%EC%A0%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏆 9단계. 도전 과제 — 뉴스 기사 분류

**목표:** 같은 경기를 다룬 두 기사(기사1, 기사2)의 유사도가 높게 나오면 성공! 🎯


---

In [1]:
# 필요한 라이브러리 설치
!apt-get install default-jdk -y
!pip install konlpy scikit-learn --quiet
print("✅ 완료!")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  at-spi2-core default-jdk-headless default-jre default-jre-headless
  fonts-dejavu-core fonts-dejavu-extra gsettings-desktop-schemas
  libatk-bridge2.0-0 libatk-wrapper-java libatk-wrapper-java-jni libatk1.0-0
  libatk1.0-data libatspi2.0-0 libxcomposite1 libxt-dev libxtst6 libxxf86dga1
  openjdk-11-jdk openjdk-11-jdk-headless openjdk-11-jre
  openjdk-11-jre-headless session-migration x11-utils
Suggested packages:
  libxt-doc openjdk-11-demo openjdk-11-source visualvm libnss-mdns
  fonts-ipafont-gothic fonts-ipafont-mincho fonts-wqy-microhei
  | fonts-wqy-zenhei fonts-indic mesa-utils
The following NEW packages will be installed:
  at-spi2-core default-jdk default-jdk-headless default-jre
  default-jre-headless fonts-dejavu-core fonts-dejavu-extra
  gsettings-desktop-schemas libatk-bridge2.0-0 libatk-wrapper-java
  libatk-wrapper-java-jn

In [2]:
import re
from konlpy.tag import Okt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

okt = Okt()
print("✅ 준비 완료!")

✅ 준비 완료!


In [3]:
# ── 2주차 전처리 함수 ──────────────────────────────────────────
불용어 = ["것","수","나","저","제","그","이","때","등","좀","잘","더","한","안","못","또"]

def 전처리(텍스트):
    텍스트 = re.sub(r'http\S+', '', 텍스트)
    텍스트 = re.sub(r'[^가-힣a-zA-Z0-9 ]', '', 텍스트)
    텍스트 = re.sub(r'(.)\u0001{2,}', r'\1\1', 텍스트).strip()
    return [w for w in okt.nouns(텍스트) if w not in 불용어 and len(w) >= 2]

print("✅ 전처리 함수 준비 완료!")

✅ 전처리 함수 준비 완료!


In [4]:
# 뉴스 기사 4개
기사1 = """LAFC가 메이저리그사커 역사에 남을 기록을 세우며 완벽한 출발을 이어가고 있다.
그러나 팀의 상승세와 달리 손흥민의 리그 득점 침묵은 계속되며 묘한 대비를 만들고 있다.
LAFC는 미국 캘리포니아 로스앤젤레스 BMO 스타디움에서 열린 2026 메이저리그사커 서부콘퍼런스 4라운드 세인트루이스 시티와의 경기에서 2-0으로 승리했다.
이 승리로 LAFC는 개막 이후 4전 전승을 기록하며 승점 12점을 확보했고 동서부 콘퍼런스를 통틀어 가장 높은 승점을 기록한 팀으로 올라섰다.
LAFC가 MLS 역사상 처음으로 시즌 첫 4경기에서 모두 무실점 승리를 기록한 팀이 됐다.
손흥민은 CONCACAF 챔피언스컵에서는 1골 4도움을 기록하며 팀 공격에 중요한 역할을 하고 있지만 정작 MLS 리그에서는 아직 득점포를 가동하지 못하고 있다.
이번 시즌 LAFC는 중원 중심의 조직적인 플레이를 강조하고 있다.
세인트루이스전에서 손흥민은 공격형 미드필더로 배치돼 나탄 오르다스의 뒤에서 공격 전개를 돕는 역할을 맡았다.
손흥민은 유럽 무대에서 오랜 기간 정상급 활약을 이어온 베테랑 공격수이며 시즌 역시 아직 초반 단계다."""

기사2 = """손흥민은 마수걸이 골 사냥에 실패했으나 소속팀 로스앤젤레스FC LAFC는 개막 4연승을 이어갔다.
LAFC는 미국 LA의 BMO 스타디움에서 열린 세인트루이스와 2026 메이저리그사커 정규리그 4라운드 홈 경기에서 후반 미드필더 마티외 슈아니에르의 멀티 골로 2-0 승리를 거뒀다.
창단 첫 리그 개막 4연승을 기록한 LAFC는 서부 콘퍼런스 선두로 올라섰다.
LAFC는 올 시즌 4경기에서 실점 없이 8골을 기록했다.
선발 출전해 후반 26분까지 뛴 손흥민은 리그 마수걸이 골 사냥에 실패했다.
지난 리그 3경기에서 손흥민은 득점 없이 도움만 3개를 기록했다.
손흥민은 아직 시즌 첫 필드골을 넣지 못했다. 챔피언스컵 득점은 페널티킥으로 올렸다.
손흥민은 2선 중앙과 최전방을 오가며 부지런히 그라운드를 누볐다.
슈아니에르가 지난해부터 뛴 LAFC에서 멀티 골을 넣은 건 이번이 처음이다."""

기사3 = "삼성전자가 신형 갤럭시 스마트폰을 출시했습니다. AI 카메라 기능과 배터리 성능이 향상됐습니다."
기사4 = "한국은행이 기준금리를 동결했습니다. 물가 안정과 경기 회복을 동시에 고려한 결정입니다."

print("기사1 [MLS 축구 - OSEN]:", 기사1[:45], "...")
print("기사2 [MLS 축구 - 연합뉴스]:", 기사2[:45], "...")
print("기사3 [스마트폰]:", 기사3[:45], "...")
print("기사4 [경제]    :", 기사4[:45], "...")

기사1 [MLS 축구 - OSEN]: LAFC가 메이저리그사커 역사에 남을 기록을 세우며 완벽한 출발을 이어가고 있다. ...
기사2 [MLS 축구 - 연합뉴스]: 손흥민은 마수걸이 골 사냥에 실패했으나 소속팀 로스앤젤레스FC LAFC는 개막 4 ...
기사3 [스마트폰]: 삼성전자가 신형 갤럭시 스마트폰을 출시했습니다. AI 카메라 기능과 배터리 성능이 ...
기사4 [경제]    : 한국은행이 기준금리를 동결했습니다. 물가 안정과 경기 회복을 동시에 고려한 결정입 ...


In [5]:
# STEP 1: 전처리 — 4개 기사 각각 명사 추출
처리_기사1 = 전처리(기사1)
처리_기사2 = 전처리(기사2)
처리_기사3 = 전처리(기사3)
처리_기사4 = 전처리(기사4)

print("전처리 결과 (명사만 추출):")
print(f"  기사1: {처리_기사1}")
print(f"  기사2: {처리_기사2}")
print(f"  기사3: {처리_기사3}")
print(f"  기사4: {처리_기사4}")

전처리 결과 (명사만 추출):
  기사1: ['메이저리그사커', '역사', '기록', '출발', '어가', '상승세', '달리', '손흥민', '리그', '득점', '침묵', '계속', '대비', '미국', '캘리포니아', '로스앤젤레스', '스타디움', '메이저리그사커', '서부', '콘퍼런스', '라운드', '세인트루이스', '시티', '경기', '승리', '승리', '개막', '이후', '전승', '기록', '승점', '확보', '서부', '콘퍼런스', '통틀어', '가장', '승점', '기록', '역사상', '처음', '시즌', '경기', '모두', '무실점', '승리', '기록', '손흥민', '챔피언스', '도움', '기록', '공격', '역할', '정작', '리그', '득점', '가동', '이번', '시즌', '중원', '중심', '조직', '플레이', '강조', '세인트루이스', '손흥민', '공격', '미드필더', '배치', '나탄', '다스', '공격', '전개', '역할', '손흥민', '유럽', '무대', '기간', '정상', '활약', '베테', '공격수', '시즌', '역시', '초반', '단계']
  기사2: ['손흥민', '마수', '걸이', '사냥', '소속', '로스앤젤레스', '개막', '연승', '미국', '스타디움', '세인트루이스', '메이저리그사커', '정규', '리그', '라운드', '경기', '후반', '미드필더', '마티외', '에르', '멀티', '골로', '승리', '창단', '리그', '개막', '연승', '기록', '서부', '콘퍼런스', '선두', '시즌', '경기', '실점', '기록', '선발', '출전', '후반', '손흥민', '리그', '마수', '걸이', '사냥', '리그', '경기', '손흥민', '득점', '도움', '기록', '손흥민', '시즌', '필드', '챔피언스', '득점', '페널티킥', '손흥민', '중앙', '전방', '라운드', '에르', '지난해', '멀티', '이번', '

In [6]:
# STEP 2: TF-IDF 행렬 생성
# 전처리 결과(리스트)를 ' '.join()으로 문자열로 변환 후 vectorizer에 전달
기사_vec = TfidfVectorizer()
기사_tfidf = 기사_vec.fit_transform([
    ' '.join(처리_기사1), ' '.join(처리_기사2),
    ' '.join(처리_기사3), ' '.join(처리_기사4)
])
기사_어휘 = 기사_vec.get_feature_names_out()
기사_배열 = 기사_tfidf.toarray()

print(f"전체 어휘: {len(기사_어휘)}개")
print(f"행렬 크기: {기사_tfidf.shape}  ← (기사 4개 × 단어 수)")

전체 어휘: 103개
행렬 크기: (4, 103)  ← (기사 4개 × 단어 수)


In [7]:
# STEP 3: 기사별 핵심어 (.argmax()로 TF-IDF 최고 단어 추출)
print("기사별 핵심어:")
print(f"  기사1 [MLS 축구 - OSEN]:      '{기사_어휘[기사_배열[0].argmax()]}'  ({기사_배열[0].max():.3f})")
print(f"  기사2 [MLS 축구 - 연합뉴스]:  '{기사_어휘[기사_배열[1].argmax()]}'  ({기사_배열[1].max():.3f})")
print(f"  기사3 [스마트폰]:              '{기사_어휘[기사_배열[2].argmax()]}'  ({기사_배열[2].max():.3f})")
print(f"  기사4 [경제]:                  '{기사_어휘[기사_배열[3].argmax()]}'  ({기사_배열[3].max():.3f})")

기사별 핵심어:
  기사1 [MLS 축구 - OSEN]:      '기록'  (0.367)
  기사2 [MLS 축구 - 연합뉴스]:  '손흥민'  (0.405)
  기사3 [스마트폰]:              '갤럭시'  (0.316)
  기사4 [경제]:                  '결정'  (0.326)


In [8]:
# STEP 4: 유사도 행렬 — 같은 주제 기사끼리 높게 나오면 성공!
기사_유사도 = cosine_similarity(기사_tfidf)

print("유사도 확인:")
print()
# 같은 주제 쌍: 높아야 함
print(f"  [같은 주제] MLS축구: 기사1 ↔ 기사2 = {기사_유사도[0][1]:.3f}  {'✅ 높음!' if 기사_유사도[0][1] > 0.3 else '🤔 낮음'}")
print()
# 다른 주제 쌍: 낮아야 함
print(f"  [다른 주제] 기사1(MLS) ↔ 기사3(스마트폰) = {기사_유사도[0][2]:.3f}  {'✅ 낮음!' if 기사_유사도[0][2] < 0.1 else '🤔 높음'}")
print(f"  [다른 주제] 기사1(MLS) ↔ 기사4(경제)    = {기사_유사도[0][3]:.3f}  {'✅ 낮음!' if 기사_유사도[0][3] < 0.1 else '🤔 높음'}")
print(f"  [다른 주제] 기사2(MLS) ↔ 기사3(스마트폰) = {기사_유사도[1][2]:.3f}  {'✅ 낮음!' if 기사_유사도[1][2] < 0.1 else '🤔 높음'}")
print(f"  [다른 주제] 기사2(MLS) ↔ 기사4(경제)    = {기사_유사도[1][3]:.3f}  {'✅ 낮음!' if 기사_유사도[1][3] < 0.1 else '🤔 높음'}")
print()
print("같은 주제끼리 유사도 높고, 다른 주제끼리 낮으면 성공! 🎯")

유사도 확인:

  [같은 주제] MLS축구: 기사1 ↔ 기사2 = 0.475  ✅ 높음!

  [다른 주제] 기사1(MLS) ↔ 기사3(스마트폰) = 0.000  ✅ 낮음!
  [다른 주제] 기사1(MLS) ↔ 기사4(경제)    = 0.025  ✅ 낮음!
  [다른 주제] 기사2(MLS) ↔ 기사3(스마트폰) = 0.000  ✅ 낮음!
  [다른 주제] 기사2(MLS) ↔ 기사4(경제)    = 0.041  ✅ 낮음!

같은 주제끼리 유사도 높고, 다른 주제끼리 낮으면 성공! 🎯
